# LLM API

Ноутбук содержит примеры использования openai библиотеки для обращения к llm.

In [1]:
import os

from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel
from dataclasses import dataclass

1. Конфигурация LLM клиента:

    Для работы с LLM необходимо создать свой персональный `OPEN_ROUTER_API_KEY` и сохранить его в .env файл в корне проекта. Создать токен можно по [ссылке](https://openrouter.ai/settings/keys).

In [2]:
load_dotenv()

BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
MODEL_NAME = "openrouter/owl-alpha"

2. Создание LLM клиента:

In [3]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

3. Запрос к LLM модели с произвольным текстом:

In [4]:
response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {"role": "user", "content": "Hello how are you?"}
    ]
)

response.output_text

"Hello! I'm doing well, thank you for asking! How can I help you today?"

4. Запрос к LLM и получение результата как structured output:

In [5]:
dataclass
class GeneratorOutput(BaseModel):
    """Модель ответа"""
    sql: str


response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {"role": "system", "content": "Your are PostgreSQL generator AI agent. Answer with only sql."},
        {"role": "user", "content": "Find all records of users that are from IT and registered after some X variable"}
    ],
    text_format=GeneratorOutput
)

response.output_parsed

GeneratorOutput(sql="SELECT * FROM users WHERE department = 'IT' AND registered_at > :X;")

5. Structured запрос и результат: 

In [6]:
dataclass
class GeneratorInput(BaseModel):
    """Модель запроса"""
    user_text: str


input = GeneratorInput(
    user_text="Find all records of users that are from IT and registered after some X variable"
)

print("Request:")
print(input.model_dump_json())

response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {"role": "system", "content": "Your are PostgreSQL generator AI agent. Answer with only sql."},
        {"role": "user", "content": input.model_dump_json()}
    ],
    text_format=GeneratorOutput
)

response.output_parsed

Request:
{"user_text":"Find all records of users that are from IT and registered after some X variable"}


GeneratorOutput(sql="SELECT * FROM users WHERE department = 'IT' AND registered_at > :X;")